Constructed examples demonstrate methods; fixed filing facts and figures are historical teaching inputs. New outputs are saved under a separate `runs/` directory. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch05_linear_algebra_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 5: covariance matrices, SVD, and PCA

The matrix and returns below are constructed teaching data, not empirical estimates. With two assets and \(w=(0.6,0.4)\), compute variance as \(w^	op\Sigma w\). Then inspect eigenvalues, rank, condition number, and a linear solve. The later PCA example uses a fixed random seed and fits its mean and components on only the first 180 observations.

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from quant_course.math_tools import pca_fit_transform

SIGMA = np.array([[0.04, 0.01], [0.01, 0.09]])
MU = np.array([0.05, 0.08])
w = np.array([0.6, 0.4])
variance = float(w @ SIGMA @ w)
assert np.isclose(variance, 0.0336)
print("variance terms:", 0.6**2 * 0.04, 2 * 0.6 * 0.4 * 0.01, 0.4**2 * 0.09)
print(f"w'Σw={variance:.8f}")


In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(SIGMA)
rank = np.linalg.matrix_rank(SIGMA)
condition = np.linalg.cond(SIGMA, p=2)
solution = np.linalg.solve(SIGMA, MU)
residual = np.linalg.norm(SIGMA @ solution - MU)
assert np.all(eigenvalues > 0)
assert rank == 2
print("eigenvalues:", eigenvalues)
print(f"PSD={np.all(eigenvalues >= 0)}, PD={np.all(eigenvalues > 0)}, rank={rank}")
print(f"2-norm condition number={condition:.8f}")
print("eigenvectors (columns):\n", eigenvectors)
print(f"solve Σx=μ gives {solution}; residual={residual:.3e}")
print("This linear-system solution is not a portfolio recommendation.")


## SVD and a past-window PCA fit

Each row is one constructed weekday return observation; each column is a fictitious asset. There are \(n=252\) rows and \(p=4\) features. The first 180 rows form the training window. PCA centers only that window, uses SVD, and computes sample explained variance with \(ddof=1\). The later 72 rows remain outside this fit and are not used to form a strategy or forecast.

In [ ]:
rng = np.random.default_rng(20261005)
n = 252
market = rng.normal(0.0, 0.012, n)
rates = rng.normal(0.0, 0.006, n)
style = rng.normal(0.0, 0.007, n)
idiosyncratic = rng.normal(0.0, [0.005, 0.007, 0.006, 0.008], size=(n, 4))
simulated = (
    market[:, None] * np.array([1.0, 0.8, 0.25, 0.1])
    + rates[:, None] * np.array([0.15, -0.2, 0.7, 0.4])
    + style[:, None] * np.array([-0.1, 0.2, -0.2, 0.5])
    + idiosyncratic
)
dates = pd.date_range("2024-01-02", periods=n, freq="B", tz="UTC")
returns = pd.DataFrame(simulated, index=dates, columns=["SIM-A", "SIM-B", "SIM-C", "SIM-D"])
train = returns.iloc[:180]
later_window = returns.iloc[180:]
pca = pca_fit_transform(train.to_numpy(), n_components=4)
assert train.index[-1] < later_window.index[0]
assert np.allclose(pca.center, train.mean(axis=0).to_numpy())
assert pca.projection.shape == (180, 4)
print(f"n={len(train)} training rows, p={train.shape[1]} return features")
print(f"train ends {train.index[-1].date()}; later window begins {later_window.index[0].date()}")
print("training center:", pca.center)
print("components shape (k, p):", pca.components.shape)
print("projection shape (n, k):", pca.projection.shape)
print("sample explained variance (ddof=1):", pca.explained_variance)
print("explained variance ratio:", pca.explained_variance_ratio)


## Interpreting dimensions

For input shape \((n,p)=(180,4)\), `center` has shape \((p,)\), `components` has shape \((k,p)\), and `projection` has shape \((n,k)\). Each projected score is a dot product between a centered observation and a component. Component signs are arbitrary; the subspace, orthogonality, and explained variance carry the invariant information.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
scatter = axes[0].scatter(pca.projection[:, 0], pca.projection[:, 1],
                          c=np.arange(len(train)), cmap="viridis", s=18, alpha=0.8)
axes[0].set_xlabel("PC1 score (return units)")
axes[0].set_ylabel("PC2 score (return units)")
axes[0].set_title("Training-window projections only")
fig.colorbar(scatter, ax=axes[0], label="Training row")
components = np.arange(1, len(pca.explained_variance_ratio) + 1)
axes[1].bar(components, pca.explained_variance_ratio, color="#4C9F70")
axes[1].set_xlabel("Principal component")
axes[1].set_ylabel("Share of training sample variance")
axes[1].set_xticks(components)
axes[1].set_title("SVD-based PCA; sample variance ddof=1")
fig.suptitle("Constructed weekday return sample; fixed seed 20261005")
fig.tight_layout()
output = RUN_DIR / "figures" / "a2" / "ch05_pca.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=160)
plt.close(fig)
output
